# Lab 03 – Formula 1 Dataset Preprocessing

This notebook loads, cleans, and prepares the Formula 1 World Championship dataset.

## 1. Import Libraries

In [1]:
import os
import pandas as pd

RAW_DIR = os.path.join('data', 'raw')
PROCESSED_DIR = os.path.join('data', 'processed')

os.makedirs(PROCESSED_DIR, exist_ok=True)

## 2. Load Raw Datasets

In [ ]:
csv_files = [
    "circuits.csv", 
    "constructor_results.csv", 
    "constructor_standings.csv",
    "constructors.csv", 
    "driver_standings.csv", 
    "drivers.csv",
    "lap_times.csv", 
    "pit_stops.csv", 
    "qualifying.csv",
    "races.csv", 
    "results.csv", 
    "seasons.csv",
    "sprint_results.csv", 
    "status.csv",
]

raw_data = {}
for file in csv_files:
    name = file.replace(".csv", "")
    path = os.path.join(RAW_DIR, file) 
    if os.path.exists(path):

        # Treat "\N" string as missing values NaN
        raw_data[name] = pd.read_csv(path, na_values="\\N")
        print(
            f"Successfully read {file:30s} | "
            f"{raw_data[name].shape[0]:>7,} rows x {raw_data[name].shape[1]} columns"
        )
    else:

        print(f"Warning: File {file} not found at {path}")

Successfully read circuits.csv                   |      77 rows x 9 columns
Successfully read constructor_results.csv        |  12,625 rows x 5 columns
Successfully read constructor_standings.csv      |  13,391 rows x 7 columns
Successfully read constructors.csv               |     212 rows x 5 columns
Successfully read driver_standings.csv           |  34,863 rows x 7 columns
Successfully read drivers.csv                    |     861 rows x 9 columns
Successfully read lap_times.csv                  | 589,081 rows x 6 columns
Successfully read pit_stops.csv                  |  11,371 rows x 7 columns
Successfully read qualifying.csv                 |  10,494 rows x 9 columns
Successfully read races.csv                      |   1,125 rows x 18 columns
Successfully read results.csv                    |  26,759 rows x 18 columns
Successfully read seasons.csv                    |      75 rows x 2 columns
Successfully read sprint_results.csv             |     360 rows x 16 columns
Successfu

## 3. Clean Individual Tables

In [3]:
# Clean drivers table
drivers = raw_data["drivers"].copy()

# Combine forename and surname for driver full name
drivers["driver_name"] = drivers["forename"] + " " + drivers["surname"]

# Parse birthdate
drivers["dob"] = pd.to_datetime(drivers["dob"], errors="coerce")

# Cast driver number to nullable integer type
drivers["number"] = pd.to_numeric(drivers["number"], errors="coerce").astype("Int64")

# Clean constructors table
constructors = raw_data["constructors"].copy()

# Clean circuits table
circuits = raw_data["circuits"].copy()
circuits["lat"] = pd.to_numeric(circuits["lat"], errors="coerce")
circuits["lng"] = pd.to_numeric(circuits["lng"], errors="coerce")
circuits["alt"] = pd.to_numeric(circuits["alt"], errors="coerce").astype("Int64")

# Clean status table and group detailed statuses
status = raw_data["status"].copy()

In [4]:
# Clean races table
races = raw_data['races'].copy()

# Combine date and time to datetime object
races['race_datetime'] = pd.to_datetime(races['date'] + ' ' + races['time'].fillna('00:00:00'), errors='coerce')
races['date'] = pd.to_datetime(races['date'], errors='coerce')

# Select important columns
races_cols = ['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'race_datetime']
races = races[races_cols]

# Clean results table
results = raw_data['results'].copy()
results['position'] = pd.to_numeric(results['position'], errors='coerce').astype('Int64')
results['milliseconds'] = pd.to_numeric(results['milliseconds'], errors='coerce').astype('Int64')
results['fastestLap'] = pd.to_numeric(results['fastestLap'], errors='coerce').astype('Int64')
results['rank'] = pd.to_numeric(results['rank'], errors='coerce').astype('Int64')
results['fastestLapSpeed'] = pd.to_numeric(results['fastestLapSpeed'], errors='coerce')

# Calculate driver's age at the time of the race
races_temp = raw_data['races'][['raceId', 'date']].copy()
races_temp['date'] = pd.to_datetime(races_temp['date'], errors='coerce')
drivers_temp = raw_data['drivers'][['driverId', 'dob']].copy()
drivers_temp['dob'] = pd.to_datetime(drivers_temp['dob'], errors='coerce')

results = results.merge(races_temp, on='raceId', how='left')
results = results.merge(drivers_temp, on='driverId', how='left')
results['driver_age'] = (results['date'] - results['dob']).dt.days / 365.25
results['driver_age'] = results['driver_age'].round(1)

# Drop temporary merge helper columns
results.drop(columns=['date', 'dob'], inplace=True)

# Clean standings and other secondary tables
driver_standings = raw_data['driver_standings'].copy()
driver_standings['points'] = pd.to_numeric(driver_standings['points'], errors='coerce')
driver_standings['position'] = pd.to_numeric(driver_standings['position'], errors='coerce').astype('Int64')
driver_standings['wins'] = pd.to_numeric(driver_standings['wins'], errors='coerce').astype('Int64')

constructor_standings = raw_data['constructor_standings'].copy()
constructor_standings['points'] = pd.to_numeric(constructor_standings['points'], errors='coerce')
constructor_standings['position'] = pd.to_numeric(constructor_standings['position'], errors='coerce').astype('Int64')
constructor_standings['wins'] = pd.to_numeric(constructor_standings['wins'], errors='coerce').astype('Int64')

qualifying = raw_data['qualifying'].copy()
qualifying['position'] = pd.to_numeric(qualifying['position'], errors='coerce').astype('Int64')

pit_stops = raw_data['pit_stops'].copy()
pit_stops['duration'] = pd.to_numeric(pit_stops['duration'], errors='coerce')
pit_stops['milliseconds'] = pd.to_numeric(pit_stops['milliseconds'], errors='coerce').astype('Int64')

lap_times = raw_data['lap_times'].copy()
lap_times['milliseconds'] = pd.to_numeric(lap_times['milliseconds'], errors='coerce').astype('Int64')

sprint_results = raw_data['sprint_results'].copy()
sprint_results['position'] = pd.to_numeric(sprint_results['position'], errors='coerce').astype('Int64')
sprint_results['milliseconds'] = pd.to_numeric(sprint_results['milliseconds'], errors='coerce').astype('Int64')

constructor_results = raw_data['constructor_results'].copy()
constructor_results['points'] = pd.to_numeric(constructor_results['points'], errors='coerce')

seasons = raw_data['seasons'].copy()


## 4. Export Cleaned & Merged Data

In [5]:
print("Exporting cleaned individual CSV files...")
cleaned_tables = {
    "circuits_cleaned.csv": circuits,
    "constructors_cleaned.csv": constructors,
    "drivers_cleaned.csv": drivers,
    "races_cleaned.csv": races,
    "results_cleaned.csv": results,
    "status_cleaned.csv": status,
    "driver_standings_cleaned.csv": driver_standings,
    "constructor_standings_cleaned.csv": constructor_standings,
    "qualifying_cleaned.csv": qualifying,
    "pit_stops_cleaned.csv": pit_stops,
    "lap_times_cleaned.csv": lap_times,
    "sprint_results_cleaned.csv": sprint_results,
    "constructor_results_cleaned.csv": constructor_results,
    "seasons_cleaned.csv": seasons,
}

for filename, df in cleaned_tables.items():
    dest_path = os.path.join(PROCESSED_DIR, filename)
    df.to_csv(dest_path, index=False)
    print(f"\t- Saved: {filename:35s} | {df.shape[0]:>7,} rows")

Exporting cleaned individual CSV files...
	- Saved: circuits_cleaned.csv                |      77 rows
	- Saved: constructors_cleaned.csv            |     212 rows
	- Saved: drivers_cleaned.csv                 |     861 rows
	- Saved: races_cleaned.csv                   |   1,125 rows
	- Saved: results_cleaned.csv                 |  26,759 rows
	- Saved: status_cleaned.csv                  |     139 rows
	- Saved: driver_standings_cleaned.csv        |  34,863 rows
	- Saved: constructor_standings_cleaned.csv   |  13,391 rows
	- Saved: qualifying_cleaned.csv              |  10,494 rows
	- Saved: pit_stops_cleaned.csv               |  11,371 rows
	- Saved: lap_times_cleaned.csv               | 589,081 rows
	- Saved: sprint_results_cleaned.csv          |     360 rows
	- Saved: constructor_results_cleaned.csv     |  12,625 rows
	- Saved: seasons_cleaned.csv                 |      75 rows
